In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Load data from the project root data/ folder
data_dir = os.path.join(os.getcwd(), '..', 'data')
file_path = os.path.join(data_dir, 'DataCoSupplyChainDataset.csv')

df = pd.read_csv(file_path, encoding='latin-1')

# Output directory for figures
fig_dir = os.path.join(os.getcwd(), '..', 'figures')
os.makedirs(fig_dir, exist_ok=True)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.info()

In [ ]:
# Fig: Global sales in different markets (paper Fig. sales_region / "Demand")
sales_data = df.groupby('Order Region')['Sales'].sum().reset_index()
plt.style.use('default')

plt.figure(figsize=(12, 6))
sns.barplot(data=sales_data, x='Order Region', y='Sales', palette='muted')
plt.xlabel('Order Region', fontsize=16)
plt.ylabel('Total Sales', fontsize=16)
plt.xticks(rotation=45, fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'total_sales_by_order_region.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig: Heatmap of normalized sales by region x category (paper Fig. sales_product / "Products.png")
heatmap_data = df.groupby(['Order Region', 'Category Name'])['Sales'].sum().unstack(fill_value=0)

scaler = MinMaxScaler()
normalized_data = scaler.fit_transform(heatmap_data)
normalized_heatmap_data = pd.DataFrame(normalized_data, columns=heatmap_data.columns, index=heatmap_data.index)

plt.figure(figsize=(14, 8))
sns.heatmap(normalized_heatmap_data, cmap='coolwarm', annot=True, fmt='.2f',
            cbar_kws={'label': 'Normalized Total Sales'},
            linewidths=.5, linecolor='black', annot_kws={'size': 12})
plt.xlabel('Category Name', fontsize=16, weight='bold')
plt.ylabel('Order Region', fontsize=16, weight='bold')
plt.xticks(rotation=90, fontsize=14)
plt.yticks(rotation=0, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'sales_heatmap_order_region_category.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig: Cumulative sales by category over time
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

time_series_data = df.groupby(['order date (DateOrders)', 'Category Name'])['Sales'].sum().reset_index()
time_series_pivot = time_series_data.pivot(index='order date (DateOrders)', columns='Category Name', values='Sales').fillna(0)
cumulative_sales = time_series_pivot.cumsum()

plt.figure(figsize=(14, 8))
sns.lineplot(data=cumulative_sales, palette='colorblind')
plt.title('Cumulative Sales by Category Name Over Time', fontsize=18)
plt.xlabel('Order Date', fontsize=14)
plt.ylabel('Cumulative Sales', fontsize=14)
plt.xticks(rotation=45)
plt.grid(visible=True, linestyle='--', alpha=0.7)
plt.legend(title='Category Name', bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol=5)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'cumulative_sales_by_category_over_time.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig: Monthly sales top 10 categories (paper Fig. sales_TS / "Sales_TS.png")
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df_filtered = df[df['order date (DateOrders)'] <= '2017-09-30']

monthly_sales_data = df_filtered.groupby(
    ['Category Name', pd.Grouper(key='order date (DateOrders)', freq='M')]
)['Sales'].sum().reset_index()

total_sales_by_category = df_filtered.groupby('Category Name')['Sales'].sum().reset_index()
top_10_categories = total_sales_by_category.nlargest(10, 'Sales')['Category Name']
monthly_sales_top_10 = monthly_sales_data[monthly_sales_data['Category Name'].isin(top_10_categories)]

plt.figure(figsize=(14, 8))
sns.lineplot(data=monthly_sales_top_10, x='order date (DateOrders)', y='Sales',
             hue='Category Name', marker='o', palette='muted')
plt.xlabel('Order Date (Monthly)', fontsize=16)
plt.ylabel('Sales', fontsize=16)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
plt.grid(visible=True, linestyle='--', alpha=1)
plt.legend(bbox_to_anchor=(0.5, 1.15), loc='upper center', ncol=5, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'monthly_sales_top_10_categories.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig: Performance comparison TF_Local vs TF_FedAvg vs TF_PA-CFL (paper Fig. 5)
import matplotlib.pyplot as plt
import numpy as np

models = ['TF_Local', 'TF_FedAvg', 'TF_PA-CFL(e=10,NC=7)']
regions = ['West of USA', 'US Center', 'West Africa', 'North Africa',
           'Central America', 'South America', 'East of USA', 'South of USA']

rmse_values = [
    [15.74, 52.66, 14.90], [15.01, 50.71, 14.13], [18.95, 59.83, 13.63],
    [18.01, 60.93, 11.19], [15.19, 53.58, 14.23], [15.88, 53.06, 12.79],
    [14.38, 51.26, 13.73], [15.50, 52.37, 12.65]
]
mae_values = [
    [12.56, 41.15, 10.52], [11.31, 39.67, 11.32], [14.85, 45.11, 11.24],
    [14.72, 47.35, 8.56],  [11.88, 42.17, 11.40], [12.76, 41.49, 10.41],
    [11.77, 40.17, 10.44], [12.91, 40.57, 10.33]
]
r2_values = [
    [98.01, 77.84, 98.22], [98.15, 78.96, 98.37], [96.87, 68.82, 98.39],
    [97.28, 69.99, 98.95], [98.14, 76.85, 98.37], [97.93, 76.88, 98.66],
    [98.31, 78.71, 98.46], [97.97, 76.68, 98.65]
]

bar_width = 0.2
x = np.arange(len(regions))
colors = ['#4C72B0', '#DD8452', '#55A868']

def create_plot(ax, data, metric):
    ax.bar(x - bar_width, [v[0] for v in data], bar_width, label=models[0], color=colors[0], edgecolor='black')
    ax.bar(x,             [v[1] for v in data], bar_width, label=models[1], color=colors[1], edgecolor='black')
    ax.bar(x + bar_width, [v[2] for v in data], bar_width, label=models[2], color=colors[2], edgecolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels(regions, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.legend(loc='lower right' if 'R' in metric else 'best', fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_facecolor('#F7F7F7')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

for data, metric, fname in [(rmse_values, 'RMSE', 'result_rmse.png'),
                              (mae_values, 'MAE', 'result_mae.png'),
                              (r2_values, 'R2 (%)', 'result_r2.png')]:
    fig, ax = plt.subplots(figsize=(6, 4))
    create_plot(ax, data, metric)
    if 'R2' in metric:
        ax.set_ylim(60, 100)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, fname), dpi=300, bbox_inches='tight')
    plt.show()